# MCP + LangGraph Hybrid Demo (Ollama)

This notebook demonstrates a **hybrid architecture**:

- An MCP-style controller (using a system prompt and JSON envelopes)
- A **local tool layer** (sum + word count)
- A **LangGraph state machine** that orchestrates:
  - Building the MCP envelope
  - Calling the controller once
  - Optionally running a tool
  - Producing a final answer

Design goals:

- **No infinite ReAct loops**
- **At most one tool call per query**
- **Transparent, debuggable flow**
- Clean separation between:
  - *Controller LLM* (JSON policy)
  - *Tools* (pure Python)
  - *Graph control* (LangGraph)


In [1]:
# If needed:
# %pip install requests langgraph

In [16]:


import json
import re
from typing import Dict, Any, List, Optional, TypedDict, Annotated
import requests

from langgraph.graph import StateGraph, END

OLLAMA_BASE = "http://localhost:11434"
MODEL = "llama3"
VERBOSE = True   # set to False to reduce console noise


In [17]:
# --- Tool registry (MCP Tool Layer) ---
TOOL_REGISTRY: Dict[str, Dict[str, Any]] = {
    "get_sum": {
        "description": "Returns the sum of two numbers",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {"type": "number", "description": "First number"},
                "b": {"type": "number", "description": "Second number"},
            },
            "required": ["a", "b"],
        },
    },
    "word_count": {
        "description": "Counts words in a string",
        "parameters": {
            "type": "object",
            "properties": {
                "text": {
                    "type": "string",
                    "description": "Text to count words from",
                }
            },
            "required": ["text"],
        },
    },
}


def exec_tool(name: str, args: Dict[str, Any]) -> Dict[str, Any]:
    """Execute an MCP-style tool locally."""
    if name == "get_sum":
        a = args.get("a")
        b = args.get("b")
        if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
            return {"error": "Invalid arguments. Expected numbers 'a' and 'b'."}
        return {"result": a + b}

    elif name == "word_count":
        text = args.get("text", "")
        if not isinstance(text, str):
            return {"error": "Invalid arguments. Expected string 'text'."}
        return {"result": len([w for w in text.split() if w])}

    else:
        return {"error": f"Unknown tool: {name}"}



In [18]:
# --- MCP layers (system/task/user/memory/tool) ---

def build_system_layer() -> str:
    return (
        "You are an assistant running under a Model Context Protocol (MCP) controller. "
        'You MUST return exactly one JSON object with key "type". '
        'When you need a tool, respond with:\n'
        '{"type":"tool_call","name":"<tool_name>","arguments":{...}}\n'
        'When you want to answer directly, respond with:\n'
        '{"type":"final","content":"<natural language answer>"}\n'
        "No extra text, no multiple JSON objects."
    )


def build_task_layer(user_query: str) -> str:
    return user_query


def build_user_layer(user_id: str = "demo_user") -> Dict[str, Any]:
    return {"id": user_id, "role": "presenter", "permissions": ["use_tools"]}


def build_memory_layer() -> List[Dict[str, Any]]:
    return [{"kind": "note", "text": "User prefers short, precise answers."}]


def build_tool_layer() -> Dict[str, Any]:
    return {"tools": TOOL_REGISTRY}


def assemble_mcp_payload(user_query: str) -> Dict[str, Any]:
    return {
        "system": build_system_layer(),
        "task": build_task_layer(user_query),
        "user": build_user_layer(),
        "memory": build_memory_layer(),
        "tool": build_tool_layer(),
    }


def build_mcp_messages(user_query: str) -> List[Dict[str, str]]:
    """Construct the messages we send to Ollama for the controller call."""
    env = assemble_mcp_payload(user_query)

    system_prompt = (
        env["system"]
        + "\n\n--- MCP CONTEXT ---\n"
        + f"User layer: {json.dumps(env['user'])}\n"
        + f"Task layer: {env['task']}\n"
        + f"Memory layer: {json.dumps(env['memory'])}\n"
        + f"Tool layer (registry): {json.dumps(env['tool'])}\n"
        + "---------------------\nReturn one JSON object only."
    )

    messages: List[Dict[str, str]] = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": env["task"]},
    ]

    if VERBOSE:
        print("\n=== MCP Envelope ===")
        print(json.dumps(env, indent=2))
        print("\n=== Controller Messages ===")
        print(json.dumps(messages, indent=2))

    return messages



In [19]:
# --- Helpers: JSON extraction + Ollama call ---

def extract_json(s: str) -> Optional[Dict[str, Any]]:
    s = (s or "").strip()
    if s.startswith("```"):
        # strip ```json fences if present
        s = re.sub(r"^```[a-zA-Z0-9]*\n", "", s)
        s = re.sub(r"\n```$", "", s)
    m = re.search(r"\{.*\}", s, flags=re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None


def call_ollama(model: str, messages: List[Dict[str, str]], want_json: bool = True) -> str:
    """Call Ollama's /api/chat endpoint, optionally requesting JSON."""
    chat_url = f"{OLLAMA_BASE}/api/chat"
    payload: Dict[str, Any] = {
        "model": model,
        "messages": messages,
        "stream": False,
        "format": "json" if want_json else None,
        "options": {"temperature": 0, "top_p": 1, "num_ctx": 4096},
    }
    payload = {k: v for k, v in payload.items() if v is not None}

    if VERBOSE:
        print("\n--- OUTBOUND to /api/chat ---")
        print(json.dumps(payload, indent=2))

    r = requests.post(chat_url, json=payload, timeout=120)
    r.raise_for_status()
    data = r.json()

    if VERBOSE:
        print("--- INBOUND from /api/chat ---")
        print(json.dumps(data, indent=2))

    # For /api/chat, content is in data["message"]["content"]
    return data["message"]["content"]


def call_controller(messages: List[Dict[str, str]]) -> Optional[Dict[str, Any]]:
    """Call the MCP controller once and parse the JSON control message."""
    raw = call_ollama(MODEL, messages, want_json=True)
    obj = extract_json(raw)

    if VERBOSE:
        print("\n--- RAW CONTROLLER OUTPUT ---")
        print(raw)
        print("\n--- PARSED JSON ---")
        print(json.dumps(obj, indent=2) if obj else "None")

    return obj



In [21]:
# --- LangGraph state + nodes (hybrid MCP controller) ---

class MCPState(TypedDict):
    question: str
    messages: List[Dict[str, str]]
    control: Optional[Dict[str, Any]]
    tool_name: Optional[str]
    tool_args: Optional[Dict[str, Any]]
    tool_result: Optional[Dict[str, Any]]
    answer: Annotated[Optional[str], "output"]


def init_node(state: MCPState) -> Dict[str, Any]:
    """Build initial MCP messages from the question."""
    msgs = build_mcp_messages(state["question"])
    return {"messages": msgs}


def controller_node(state: MCPState) -> Dict[str, Any]:
    """Call the controller once to decide: tool_call or final."""
    ctrl = call_controller(state["messages"])

    if not ctrl or "type" not in ctrl:
        return {"answer": "Controller returned invalid or empty response."}

    typ = ctrl.get("type")
    updates: Dict[str, Any] = {"control": ctrl}

    if typ == "tool_call":
        updates["tool_name"] = ctrl.get("name")
        updates["tool_args"] = ctrl.get("arguments", {})
        if VERBOSE:
            print(f"\n[Controller] Decided to call tool: {updates['tool_name']} "
                  f"with args={updates['tool_args']}")
    elif typ == "final":
        updates["answer"] = ctrl.get("content", "")
        if VERBOSE:
            print("\n[Controller] Returned final answer directly.")
    else:
        updates["answer"] = f"Unexpected controller type: {typ}"

    return updates


def tool_node(state: MCPState) -> Dict[str, Any]:
    """Execute the chosen tool (if any)."""
    ctrl = state.get("control") or {}
    if ctrl.get("type") != "tool_call":
        if VERBOSE:
            print("\n[Tool Node] No tool_call in controller response. Skipping.")
        return {}

    name = state.get("tool_name")
    args = state.get("tool_args") or {}

    if VERBOSE:
        print(f"\n[Tool Node] Running tool '{name}' with args={args}")

    res = exec_tool(name, args)

    if VERBOSE:
        print("[Tool Node] Tool result:", res)

    return {"tool_result": res}


def final_node(state: MCPState) -> Dict[str, Any]:
    """
    Produce the final natural-language answer.

    If the controller already gave a 'final' content, we keep it.
    Otherwise, we build a short answer from the tool_result.
    """
    if state.get("answer"):
        # Answer was already provided by the controller LLM
        if VERBOSE:
            print("\n[Final Node] Using controller-provided final answer.")
        return {}

    ctrl = state.get("control") or {}
    if ctrl.get("type") == "tool_call":
        name = state.get("tool_name")
        res = state.get("tool_result") or {}

        if "error" in res:
            text = f"Tool '{name}' returned an error: {res['error']}"
        else:
            text = f"The result from tool '{name}' is: {res.get('result')}"

        if VERBOSE:
            print("\n[Final Node] Constructed final answer from tool result.")

        return {"answer": text}

    # Fallback
    if VERBOSE:
        print("\n[Final Node] No answer and no tool result; using generic fallback.")
    return {"answer": "No answer could be produced."}


# Build the LangGraph
graph = StateGraph(MCPState)

graph.add_node("init", init_node)
graph.add_node("controller", controller_node)
graph.add_node("tool", tool_node)
graph.add_node("final", final_node)

graph.set_entry_point("init")
graph.add_edge("init", "controller")
graph.add_edge("controller", "tool")
graph.add_edge("tool", "final")
graph.add_edge("final", END)

mcp_agent = graph.compile()




In [22]:
def run_mcp_demo(question: str) -> str:
    print("\n==============================")
    print(f"Question: {question}")
    print("==============================")

    initial_state: MCPState = {
        "question": question,
        "messages": [],
        "control": None,
        "tool_name": None,
        "tool_args": None,
        "tool_result": None,
        "answer": None,
    }

    result = mcp_agent.invoke(initial_state)

    print("\n=== FINAL ANSWER ===")
    print(result["answer"])
    print("====================\n")

    return result["answer"]


## Try it
Uncomment one of the following calls to see the full MCP loop with verbose logs.

In [23]:
 run_mcp_demo("What's the sum of 12 and 30?")
# run_mcp_demo("Count words in: 'This is a small MCP demo with Ollama.'")


Question: What's the sum of 12 and 30?

=== MCP Envelope ===
{
  "system": "You are an assistant running under a Model Context Protocol (MCP) controller. You MUST return exactly one JSON object with key \"type\". When you need a tool, respond with:\n{\"type\":\"tool_call\",\"name\":\"<tool_name>\",\"arguments\":{...}}\nWhen you want to answer directly, respond with:\n{\"type\":\"final\",\"content\":\"<natural language answer>\"}\nNo extra text, no multiple JSON objects.",
  "task": "What's the sum of 12 and 30?",
  "user": {
    "id": "demo_user",
    "role": "presenter",
    "permissions": [
      "use_tools"
    ]
  },
  "memory": [
    {
      "kind": "note",
      "text": "User prefers short, precise answers."
    }
  ],
  "tool": {
    "tools": {
      "get_sum": {
        "description": "Returns the sum of two numbers",
        "parameters": {
          "type": "object",
          "properties": {
            "a": {
              "type": "number",
              "description": "Fi

"The result from tool 'get_sum' is: 42"

In [24]:
run_mcp_demo("Count words in: 'This is a small MCP demo with Ollama.'")


Question: Count words in: 'This is a small MCP demo with Ollama.'

=== MCP Envelope ===
{
  "system": "You are an assistant running under a Model Context Protocol (MCP) controller. You MUST return exactly one JSON object with key \"type\". When you need a tool, respond with:\n{\"type\":\"tool_call\",\"name\":\"<tool_name>\",\"arguments\":{...}}\nWhen you want to answer directly, respond with:\n{\"type\":\"final\",\"content\":\"<natural language answer>\"}\nNo extra text, no multiple JSON objects.",
  "task": "Count words in: 'This is a small MCP demo with Ollama.'",
  "user": {
    "id": "demo_user",
    "role": "presenter",
    "permissions": [
      "use_tools"
    ]
  },
  "memory": [
    {
      "kind": "note",
      "text": "User prefers short, precise answers."
    }
  ],
  "tool": {
    "tools": {
      "get_sum": {
        "description": "Returns the sum of two numbers",
        "parameters": {
          "type": "object",
          "properties": {
            "a": {
          

"The result from tool 'word_count' is: 8"